# Setup for exercise B

In [ ]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
img_name = input("Image name: ")
read_img = cv.imread(f"{img_name}.jpg", cv.IMREAD_GRAYSCALE)
if read_img is None:
    raise FileNotFoundError(f"Image \"{img_name}.jpg\" not found.")
img: np.ndarray = read_img
img_width = img.shape[1]
img_height = img.shape[0]


# B1

In [ ]:
def three_edge_detect_methods(image: np.ndarray, base_image_name: str) -> bool:
    """Speed up a repetitive process used in exercises B1 and B2.
    Put the original image through 3 edge detection methods, then display it with the 3 resulting images.
    Input image must be np.uint8."""
    if image.dtype != np.uint8:
        raise TypeError("Input image is not of type numpy.uint8")
    img_laplacian = cv.Laplacian(src=image.astype(np.float32), ddepth=-1, ksize=5)
    img_laplacian = np.clip(img_laplacian, 0, 255)
    img_sobel = cv.Sobel(src=image.astype(np.float32), ddepth=-1, dx=1, dy=1, ksize=5)
    img_sobel = np.abs(img_sobel) # since the Sobel method can output negative values
    img_sobel = np.clip(img_sobel, 0, 255)
    img_canny = cv.Canny(image=image, threshold1=64, threshold2=128, apertureSize=3)
    processed_images: list[tuple[str, np.ndarray]] = [(f"{base_image_name}", image),
    ("Laplacian", img_laplacian),
    ("Sobel", img_sobel),
    ("Canny", img_canny)]
    plt.figure(figsize=(16, 6))
    for idx, (name, image) in  enumerate(processed_images, start=1):
        plt.subplot(1, 4, idx)
        plt.title(name)
        plt.imshow(image, cmap="gray")
        plt.axis("off")
    plt.show()
    return True


In [ ]:
three_edge_detect_methods(img, base_image_name="Original")


# B2

In [ ]:
def add_gaussian_noise(image: np.ndarray, mean: float = 0, stddev: float = 30):
    """Add Gaussian noise to an image. Input must be np.uint8, output is also np.uint8."""
    noise = np.random.normal(loc=mean, scale=stddev, size=image.shape).astype(np.float32)
    output = np.clip(image + noise, 0, 255)
    return output.astype(np.uint8)

img_noisy = add_gaussian_noise(img, 0, 30)
three_edge_detect_methods(img_noisy, base_image_name="Noisy image")
blur_klength = 5
img_smooth = cv.GaussianBlur(src=img_noisy, ksize=[blur_klength] * 2, sigmaX=0, sigmaY=0)
three_edge_detect_methods(img_smooth, base_image_name="Noisy image - blurred")


## Observations
*The "Result" columns report on improvement of the result after applying Gaussian blur compared to before the application.*
| Kernel size of Gaussian blur | Result for Laplacian method | Result for Sobel method | Result for Canny method |
|:-:|-|-|-|
| 3 | Decent | Minimal improvement | Pretty much doesn't help |
| 5 | Good | Still barely better than ksize=3 | Acceptable |
| 7 | Slightly better than ksize=5, also less noise | Same as above | Loses too many edges |

# B3

In [ ]:
thres1 = 60
thres2 = 120
images = [img,
cv.threshold(src=img, thresh=thres1, maxval=255, type=cv.THRESH_BINARY)[1],
cv.threshold(src=img, thresh=thres2, maxval=255, type=cv.THRESH_BINARY)[1],
cv.threshold(src=img, thresh=0, maxval=255, type=cv.THRESH_BINARY + cv.THRESH_OTSU)[1]] # just cv.THRESH_OTSU works, but that's how it's written in the docs
image_labels = ["Original", f"Threshold = {thres1}", f"Threshold = {thres2}", "Automatic threshold using Otsu's algorithm"]

plt.figure(figsize=(16, 6))
for idx, (image, label) in enumerate(zip(images, image_labels), start=1):
    plt.subplot(1, 4, idx)
    plt.imshow(image, cmap="gray")
    plt.title(label)
    plt.axis("off")
plt.show()


# B4

*For learning purposes, the functions in this task are implemented manually by me.*

In [ ]:
# Both functions assume radius of str_elem = 1 and zero padding at borders.
def erode(image: np.ndarray, str_elem: np.ndarray) -> np.ndarray:
    output = np.zeros(shape=image.shape)
    image = np.pad(array=image, pad_width=(1,1), mode="constant", constant_values=(0,0))
    height, width = image.shape
    for y in range(1, height - 1): 
        for x in range(1, width - 1):
            neighborhood = image[y-1:y+2, x-1:x+2]
            # Check if all on pixels of str_elem cover on pixels of the image (but NOT the other way around!)
            output[y-1][x-1] = np.all(neighborhood[str_elem == 1] == 1)
    return output

def dilate(image: np.ndarray, str_elem: np.ndarray) -> np.ndarray:
    output = np.zeros(shape=image.shape)
    image = np.pad(array=image, pad_width=(1,1), mode="constant", constant_values=(0,0))
    height, width = image.shape
    for y in range(1, height - 1): 
        for x in range(1, width - 1):
            neighborhood = image[y-1:y+2, x-1:x+2]
            # Check if any on pixels of str_elem cover an on pixel of the image (but NOT the other way around!)
            output[y-1][x-1] = np.any(neighborhood[str_elem == 1] == 1)
    return output


In [ ]:
base_images = [[[0]*8,
[0]*4 + [1]*3 + [0],
[0]*2 + [1]*4 + [0]*2,
[0]*1 + [1]*4 + [0]*3,
[0]*1 + [1]*4 + [0]*3,
[0]*1 + [1]*3 + [0]*4,
[0]*2 + [1]*2 + [0]*4,
[0]*8],
[[0]*4 + [1]*3 + [0],
[0]*4 + [1]*3 + [0],
[0]*4 + [1]*3 + [0],
[0]*3 + [1]*2 + [0]*3,
[0]*1 + [1]*3 + [0]*4,
[0]*1 + [1]*3 + [0]*4,
[0]*1 + [1]*3 + [0]*4,
[0]*8]] # 0: image in part A of this labwork, 1: an image in lecture 10


In [ ]:
img_A = np.array(base_images[1])
struct_element = np.array([[0, 1, 0],
[1]*3,
[0, 1, 0]])


In [ ]:
images: list[np.ndarray] = [img_A, erode(img_A, struct_element), dilate(img_A, struct_element)]
# open = erode than dilate
images.append(dilate(images[1], struct_element))
# close = dilate than erode
images.append(erode(images[2], struct_element))
image_labels: list[str] = ["Original", "Eroded", "Dilated", "Opened", "Closed"]

plt.figure(figsize=(15, 5))
for idx, (image, label) in enumerate(zip(images, image_labels), start=1):
    plt.subplot(1, 5, idx)
    plt.title(label)
    plt.imshow(image, cmap="gray_r") # invert colors so that foreground object is black and background is white
    # Draw grid for image and align it with the pixels
    plt.grid(True)
    plt.xticks(np.arange(image.shape[1]+1) - 0.5)
    plt.yticks(np.arange(image.shape[0]+1) - 0.5)
    plt.tick_params(labelbottom=False, labelleft=False)
plt.tight_layout()
plt.show()
